# Bài 24: Hình học xạ ảnh

Bài 8 đã xây dựng một hệ phân cấp các phép biến đổi 2D &mdash; Euclid, đồng dạng, affine &mdash; và lưu ý rằng cả ba đều bảo toàn các đường song song. Bài này trình bày bước tiếp theo, tổng quát nhất: **phép biến đổi xạ ảnh (projective transform) hay phép đồng nhất (homography)**, mô hình hóa điều mà một camera thực sự làm khi nhìn vào một bề mặt phẳng từ một góc nghiêng, và *không* bảo toàn tính song song. Đó không phải lỗi &mdash; đó chính là hiện tượng điểm tụ (vanishing point), và là công cụ đằng sau việc hiệu chỉnh phối cảnh và ghép ảnh.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Tọa độ thuần nhất

Biểu diễn một điểm 2D $(x, y)$ dưới dạng vector 3 chiều $(x, y, 1)$. Khi đó mọi ma trận $3\times3$ $H$ đều có thể tác động lên nó bằng phép nhân ma trận thông thường; chuyển về 2D nghĩa là chia cho tọa độ thứ ba:

$$\begin{bmatrix}x'\\y'\\w'\end{bmatrix} = H\begin{bmatrix}x\\y\\1\end{bmatrix}, \qquad (x_{\text{2D}}', y_{\text{2D}}') = \left(\frac{x'}{w'}, \frac{y'}{w'}\right)$$

Có hai lợi ích lớn: thứ nhất, một ma trận $3\times3$ cũng có thể biểu diễn phép tịnh tiến (điều không thể với chỉ một ma trận $2\times2$, đó là lý do Bài 8 cần một vector tịnh tiến $t$ riêng). Thứ hai, phép chia cho $w'$ chính là thứ cho phép một phép đồng nhất mô hình hóa các hiệu ứng *phối cảnh* thực sự &mdash; không chỉ các phép affine ở Bài 8, mà cả những bức ảnh nơi các đường song song hội tụ về một điểm tụ, đúng như điều một camera thực làm khi nhìn vào một bề mặt phẳng từ một góc nghiêng. Cũng lưu ý rằng $H$ và $cH$ (bất kỳ bội số vô hướng khác không nào) biểu diễn *chính xác cùng một* phép biến đổi, vì phép chia triệt tiêu hệ số tỉ lệ &mdash; một phép đồng nhất chỉ có 8 bậc tự do độc lập, không phải 9.

## Các đường song song không còn song song

Hệ phân cấp được cập nhật, mở rộng bảng ở Bài 8:

| Phép biến đổi | Dạng ma trận | Bảo toàn |
|---|---|---|
| Affine | $\begin{bmatrix}a&b&t_x\\c&d&t_y\\0&0&1\end{bmatrix}$ | tính song song |
| **Xạ ảnh** | $\begin{bmatrix}a&b&t_x\\c&d&t_y\\g&h&1\end{bmatrix}$ | chỉ đường thẳng |

Khác biệt duy nhất là hàng dưới cùng khác không $(g, h)$ &mdash; và chỉ điều đó thôi đã đủ để phá hủy tính song song. Ta minh họa trực tiếp: lấy hai đường thẳng đứng song song (hãy nghĩ đến hai thanh ray của đường tàu hỏa) và áp dụng một phép đồng nhất có hàng dưới cùng khác không.

In [ ]:
H = np.array([
    [-3, 0,  2],
    [0,  2, -1],
    [0,  1,  1],
], dtype=np.float64)

def apply_homography(points, H):
    homogeneous = np.hstack([points, np.ones((len(points), 1))])
    transformed = (H @ homogeneous.T).T
    return transformed[:, :2] / transformed[:, 2:3]

line1 = np.array([[0, 0], [0, 1]], dtype=np.float64)   # x = 0
line2 = np.array([[1, 0], [1, 1]], dtype=np.float64)   # x = 1, parallel to line1

l1_transformed = apply_homography(line1, H)
l2_transformed = apply_homography(line2, H)

print('line 1, transformed:', l1_transformed)
print('line 2, transformed:', l2_transformed)

def extended(p1, p2, back, fwd):
    direction = (p2 - p1)
    return p1 - back * direction, p2 + fwd * direction

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

for line, color in [(line1, 'tab:blue'), (line2, 'tab:orange')]:
    a, b = extended(line[0], line[1], back=3, fwd=2)
    axes[0].plot([a[0], b[0]], [a[1], b[1]], color=color)
    axes[0].plot(line[:, 0], line[:, 1], 'o', color=color)
axes[0].set_xlim(-3, 3)
axes[0].set_ylim(-3, 3)
axes[0].set_title('Before: two parallel lines')

# Each transformed line is itself exactly straight, so extending it far in BOTH
# directions (using the line's own direction, not the pre-image's) and letting
# the axes clip it is what actually spans edge to edge -- a small fixed "back"
# offset in pre-image space corresponds to a very different, non-uniform step
# once warped, since a homography preserves straightness but not spacing.
for line, color in [(l1_transformed, 'tab:blue'), (l2_transformed, 'tab:orange')]:
    a, b = extended(line[0], line[1], back=60, fwd=60)
    axes[1].plot([a[0], b[0]], [a[1], b[1]], color=color)
    axes[1].plot(line[:, 0], line[:, 1], 'o', color=color)
axes[1].set_xlim(-3, 3)
axes[1].set_ylim(-3, 3)
axes[1].set_title('After: a homography with a nonzero bottom row')

plt.tight_layout()
plt.show()

## Điểm tụ, tính bằng hai cách

Hai đường thẳng này thực sự gặp nhau ở đâu? Ta có thể tìm theo hai cách: giao trực tiếp các đoạn thẳng đã biến đổi, hoặc &mdash; thanh lịch hơn &mdash; biến đổi *điểm ở vô cực (point at infinity)* theo hướng chung $(0, 1)$ của các đường thẳng gốc, được biểu diễn trong tọa độ thuần nhất là $(0, 1, 0)$ (tọa độ thứ ba khác không sẽ làm nó thành một điểm hữu hạn; bằng không nghĩa là "ở xa vô cùng"). Áp dụng $H$ lên điểm ở vô cực đó sẽ rơi đúng vào điểm tụ.

In [ ]:
def line_intersection(p1, p2, p3, p4):
    A = np.array([[p2[0] - p1[0], -(p4[0] - p3[0])],
                  [p2[1] - p1[1], -(p4[1] - p3[1])]])
    b = np.array([p3[0] - p1[0], p3[1] - p1[1]])
    t = np.linalg.solve(A, b)[0]
    return p1 + t * (p2 - p1)

vanishing_point_direct = line_intersection(l1_transformed[0], l1_transformed[1],
                                            l2_transformed[0], l2_transformed[1])

point_at_infinity = np.array([0.0, 1.0, 0.0])  # direction (0,1), infinitely far away
transformed_infinity = H @ point_at_infinity
vanishing_point_via_infinity = transformed_infinity[:2] / transformed_infinity[2]

print('vanishing point (line intersection): ', vanishing_point_direct)
print('vanishing point (H @ infinity trick):', vanishing_point_via_infinity)

## Đường thẳng trong tọa độ thuần nhất

Điểm không phải là thứ duy nhất mà tọa độ thuần nhất biểu diễn một cách thanh lịch &mdash; một đường thẳng $ax+by+c=0$ cũng tự nhiên không kém là vector 3 chiều $l=(a,b,c)$, với một điểm $p=(x,y,1)$ nằm trên nó khi và chỉ khi tích vô hướng $l\cdot p= l^\top p = 0$. Hai hệ quả hữu ích suy ra từ tích có hướng (cross product): đường thẳng đi qua hai điểm là $l = p_1\times p_2$ (nó trực giao với cả hai, nên cả hai tích vô hướng đều triệt tiêu), và theo cùng logic theo chiều ngược lại, giao điểm của hai đường thẳng là $p = l_1\times l_2$. `line_intersection` ở trên đã giải tay một hệ tuyến tính $2\times2$; tích có hướng làm cùng việc đó chỉ trong một dòng code.

In [ ]:
def line_through(p1, p2):
    return np.cross([p1[0], p1[1], 1.0], [p2[0], p2[1], 1.0])

def intersect(l1, l2):
    p = np.cross(l1, l2)
    return p[:2] / p[2]

line1_h = line_through(l1_transformed[0], l1_transformed[1])
line2_h = line_through(l2_transformed[0], l2_transformed[1])
vanishing_point_cross = intersect(line1_h, line2_h)

print('vanishing point (cross-product trick):', vanishing_point_cross)

Khớp chính xác với hai cách trước. Điểm và đường thẳng (trong mặt phẳng 2D) hóa ra đóng vai trò hoàn toàn đối xứng trong tọa độ thuần nhất &mdash; cùng một phép tích có hướng dựng một đường thẳng từ hai điểm hoặc một điểm từ hai đường thẳng.

Ảnh động này quét phép đồng nhất liên tục từ phép đồng nhất đơn vị (các đường thẳng vẫn song song) tới $H$ đầy đủ ở trên, tính lại điểm tụ theo cả hai cách ở mỗi khung hình &mdash; bằng cách giao các đường thẳng đã biến đổi hiện tại, và bằng cách áp dụng phép đồng nhất (một phần) hiện tại lên điểm ở vô cực $(0,1,0)$. Hai cách không bao giờ bất đồng, tại bất kỳ điểm nào trong quá trình quét, xác nhận rằng đẳng thức đúng với *mọi* phép đồng nhất, không chỉ phép cuối cùng.

<img src="../animations/vanishing_point_demo.gif" alt="Ảnh động quét phép đồng nhất từ đơn vị tới một phép biến đổi xạ ảnh đầy đủ, cho thấy hai đường thẳng đứng song song hội tụ như đường ray tàu về một điểm tụ, được tính cả bằng giao đường thẳng lẫn bằng biến đổi điểm ở vô cực" style="display: block; margin: 0 auto; max-width: 680px;">

## Khi nào phép đồng nhất thực sự áp dụng được?

Một phép đồng nhất liên hệ chính xác hai bức ảnh trong đúng hai tình huống: chụp một bề mặt phẳng từ hai vị trí khác nhau (trường hợp dùng bên dưới), hoặc chụp *bất kỳ* cảnh 3D nào, phẳng hay không, từ cùng một vị trí camera trong khi chỉ xoay camera giữa các lần chụp. Trường hợp thứ hai chính là điều giúp việc ghép ảnh toàn cảnh ở Bài 25 hoạt động trên các cảnh thông thường, không phẳng &mdash; một camera tịnh tiến hoặc một cảnh thực sự 3D nhìn từ hai vị trí khác nhau thì không có phép đồng nhất duy nhất nào liên hệ chính xác hai ảnh.

## Hiệu chỉnh phối cảnh: nắn thẳng một tài liệu được chụp

Ứng dụng thực tế phổ biến nhất của phép đồng nhất: cho 4 cặp điểm tương ứng (ví dụ 4 góc của một tài liệu, do người dùng nhấp chọn hoặc tìm tự động), `cv2.getPerspectiveTransform` giải ra phép đồng nhất duy nhất ánh xạ một tập 4 điểm sang tập kia, và `cv2.warpPerspective` áp dụng nó. Hình thang mà một hình chữ nhật được chụp lại mang &mdash; các cạnh vốn song song ngoài đời thực hội tụ thấy rõ trong ảnh &mdash; được gọi là **hiệu ứng keystone (keystoning)**, và đó chính là hiệu ứng điểm tụ ở trên, giờ áp dụng lên một tài liệu thay vì một cặp đường thẳng trừu tượng. (Bên dưới, `H_rectify` được tính từ các cặp điểm tương ứng theo thứ tự ngược lại, nhưng nó chính xác là `np.linalg.inv(H_distort)`, vì hoàn tác một phép đồng nhất và nghịch đảo ma trận của nó là cùng một phép toán.)

In [ ]:
document = np.full((300, 220, 3), 255, dtype=np.uint8)
cv2.rectangle(document, (20, 20), (200, 280), (0, 0, 0), 3)
cv2.putText(document, 'HELLO', (30, 150), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 3)

h, w = document.shape[:2]
corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
photographed_corners = np.float32([[45, 15], [175, 25], [210, 290], [5, 280]])  # strong keystone: top edge narrower than bottom

H_distort = cv2.getPerspectiveTransform(corners, photographed_corners)
photographed = cv2.warpPerspective(document, H_distort, (w, h))

H_rectify = cv2.getPerspectiveTransform(photographed_corners, corners)
rectified = cv2.warpPerspective(photographed, H_rectify, (w, h))

fig, axes = plt.subplots(1, 3, figsize=(9, 4))
for ax, im, title in zip(axes, [document, photographed, rectified],
                          ['Original document', 'Photographed at an angle\n(simulated)', 'Rectified\n(from 4 corner clicks)']):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

error = np.abs(rectified.astype(int) - document.astype(int))
print(f'mean abs pixel difference after distort-then-rectify round trip: {error.mean():.2f}  (small resampling loss only)')

Ảnh động này cho thấy tài liệu biến dạng thành hình keystone, rồi được nắn thẳng từ tọa độ các góc.

<img src="../animations/document_rectification_demo.gif" alt="Ảnh động biến dạng phối cảnh: 4 góc của một tài liệu trượt tới vị trí ảnh chụp bị keystone rồi quay lại, với ảnh biến dạng trực tiếp qua phép đồng nhất giải từ 4 cặp tương ứng đó ở mỗi khung hình" style="display: block; margin: 0 auto; max-width: 680px;">

## Giải phép đồng nhất từ nhiều cặp tương ứng

4 cặp tương ứng xác định chính xác 8 bậc tự do của một phép đồng nhất, không còn dư địa cho sai số. Với *nhiều hơn* 4 cặp (thường từ so khớp đặc trưng tự động, Bài 20), bài toán trở thành một phép khớp bình phương tối thiểu thừa ràng buộc &mdash; thuật toán Biến đổi Tuyến tính Trực tiếp (Direct Linear Transform — DLT) mà `cv2.findHomography` cài đặt, có thể được bọc trong RANSAC (Bài 23) để loại bỏ các cặp tương ứng xấu (đặc trưng khớp nhầm) như ngoại lai. `cv2.findHomography` cũng chuẩn hóa tọa độ điểm bên trong trước khi giải, để cải thiện điều kiện số &mdash; cùng mẹo được làm tường minh trong thuật toán 8 điểm ở Bài 28.

In [ ]:
rng = np.random.default_rng(0)
H_true = np.array([[1, 0.2, 10], [0.05, 1, 5], [0.0008, 0.0003, 1]])

pts1 = rng.uniform(0, 200, (30, 2))
pts2 = apply_homography(pts1, H_true)

H_estimated, inlier_mask = cv2.findHomography(pts1, pts2, cv2.RANSAC, 3.0)

print('true homography (scale-normalized):\n', np.round(H_true / H_true[2, 2], 4))
print('estimated homography:\n', np.round(H_estimated, 4))
print(f'inliers: {int(inlier_mask.sum())} / {len(inlier_mask)}')

Với các cặp tương ứng sạch, `findHomography` khôi phục `H_true` gần như chính xác &mdash; đây chính là bước cuối cùng của một quy trình ghép ảnh điển hình: phát hiện và so khớp đặc trưng SIFT giữa hai bức ảnh chồng lấn (Bài 20), rồi giải ra phép đồng nhất căn chỉnh ảnh này lên ảnh kia.

## RANSAC trong thực tế: ước lượng phép đồng nhất với các cặp khớp xấu

Dù đoạn code ở trên truyền `cv2.RANSAC` cho `findHomography`, với các cặp tương ứng sạch thì chẳng có gì để nó loại bỏ &mdash; nó không tạo ra khác biệt nào thấy được. Vậy RANSAC thực sự có tác dụng gì? Vì so khớp đặc trưng thực tế (Bài 20) chắc chắn tạo ra các đặc trưng khớp nhầm, ngay cả sau phép kiểm tra tỉ số, ta cần một cách để loại bỏ các ngoại lai như vậy. Để minh họa, ta chèn 15 cặp tương ứng rác có chủ đích cùng với 30 cặp thật và so sánh phép khớp RANSAC với phép khớp bình phương tối thiểu đơn thuần trên chính dữ liệu bị nhiễm bẩn đó.

In [ ]:
pts1_bad = rng.uniform(0, 200, (15, 2))
pts2_bad = rng.uniform(0, 300, (15, 2))  # unrelated -- simulated bad matches

pts1_contaminated = np.vstack([pts1, pts1_bad])
pts2_contaminated = np.vstack([pts2, pts2_bad])

H_ransac, mask = cv2.findHomography(pts1_contaminated, pts2_contaminated, cv2.RANSAC, 3.0)
H_plain, _ = cv2.findHomography(pts1_contaminated, pts2_contaminated, 0)  # method=0: plain least squares, no outlier rejection

def mean_reprojection_error(H, pts1, pts2):
    return np.linalg.norm(apply_homography(pts1, H) - pts2, axis=1).mean()

print(f'inliers found by RANSAC: {int(mask.sum())} / {len(mask)}  (30 correspondences were genuinely correct)')
print()
print(f'mean reprojection error on the TRUE inliers:')
print(f'  RANSAC fit: {mean_reprojection_error(H_ransac, pts1, pts2):.2e} pixels')
print(f'  plain fit:  {mean_reprojection_error(H_plain, pts1, pts2):.2e} pixels')

RANSAC tìm ra đúng 30 cặp tương ứng thật và tái tạo phép đồng nhất với độ chính xác về cơ bản ngang độ chính xác máy. Phép khớp bình phương tối thiểu đơn thuần, với cùng dữ liệu, có sai số lớn hơn nhiều bậc độ lớn &mdash; về thực chất là vô dụng cho bất cứ việc gì đòi hỏi độ chính xác cấp điểm ảnh &mdash; vì 15 cặp khớp xấu trên 45 (33%) là quá đủ để làm hỏng đáng kể một phép khớp tổng bình phương không có trọng số.

## Nhìn về phía trước: hình học xạ ảnh trong 3D

Mọi thứ trong bài này đều là 2D-sang-2D: một phép đồng nhất liên hệ hai mặt phẳng (các ảnh hoặc các bề mặt phẳng trong thế giới thực). Bài 26 mở rộng cùng bộ công cụ xạ ảnh đó thêm một chiều, suy ra ma trận chiếu của camera ánh xạ một cảnh 3D lên một ảnh 2D ngay từ đầu.

### Bài tập

1. Thêm nhiễu Gauss trung bình 0 (ví dụ độ lệch chuẩn 2 điểm ảnh) vào `pts2` trước khi gọi `cv2.findHomography`. Phép đồng nhất ước lượng trôi khỏi `H_true` bao nhiêu, và việc tăng số cặp tương ứng (chẳng hạn từ 30 lên 200) có làm giảm độ trôi đó không?
2. Tăng số cặp khớp xấu được chèn (`pts1_bad`/`pts2_bad`) từ 15 cho đến khi RANSAC bắt đầu thất bại. Dùng công thức số vòng lặp ở Bài 23 (kích thước mẫu tối thiểu 4 cho phép đồng nhất), đó là tỉ lệ ngoại lai khoảng bao nhiêu, và về lý thuyết cần bao nhiêu vòng lặp để giữ độ tin cậy thành công 99%?
3. Trong ví dụ điểm tụ, đổi hai đường thẳng thành *nằm ngang* thay vì thẳng đứng (ví dụ `y=0` và `y=1`), và dự đoán, rồi kiểm chứng, điểm tụ của chúng nằm ở đâu bằng mẹo điểm-ở-vô-cực với hướng `(1, 0, 0)`.